# Leakage-safe one-hour-ahead Linear Regression and XGBoost

This notebook explains and reviews the reproducible outputs created by `src/train_one_step_models.py`. It does not refit models or alter the feature master.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
TABLES = ROOT / 'results' / 'models' / 'tables'
MODELS = ROOT / 'models' / 'one_step'
pd.set_option('display.max_columns', 20)

## Why chronological splits are mandatory

Demand forecasting must imitate deployment: models learn from earlier years and face later, unseen periods. Random splitting would let nearby hours from the future influence training and would produce an overly optimistic estimate. The fixed UTC periods are training (2022–2023), validation (January–June 2024), and test (July–December 2024).

In [ ]:
eligibility = pd.read_csv(TABLES / 'model_eligibility_summary.csv')
eligibility.query("algorithm == 'linear_regression'")[[
    'split', 'feature_group', 'split_total_rows', 'eligible_rows', 'common_comparison_rows'
]].reset_index(drop=True)

## Training-only scaling

Linear Regression is stored as a pipeline: `StandardScaler` followed by `LinearRegression`. The scaler is fitted only on eligible training predictors. Validation and test predictors are transformed with those fixed training means and scales. Targets from validation or test never enter scaling.

In [ ]:
linear_metadata = json.loads((MODELS / 'linear_regression__autoregressive_demand.metadata.json').read_text())
{key: linear_metadata[key] for key in [
    'fit_split', 'scaler_fit_split', 'training_row_count',
    'scaler_training_row_count', 'pipeline_steps'
]}

## Validation selects XGBoost settings

Every candidate is fitted on training rows only. Validation MAE chooses one fixed candidate per feature group. The test set remains untouched until those choices are complete, because repeatedly adjusting settings after seeing test results would turn the test set into another validation set.

In [ ]:
search = pd.read_csv(TABLES / 'xgboost_validation_search.csv')
search.loc[search['selected'], [
    'feature_group', 'candidate_id', 'n_estimators', 'max_depth',
    'learning_rate', 'subsample', 'colsample_bytree',
    'min_child_weight', 'reg_lambda', 'mae_mwh'
]].reset_index(drop=True)

## Natural eligibility and common comparison subsets

Natural metrics use every row eligible for that particular feature group. This is transparent, but different row sets can make comparisons unfair. Common metrics restrict every model and persistence to the exact same validation timestamps and the exact same test timestamps. Common-subset validation MAE is therefore the primary comparison.

In [ ]:
natural = pd.read_csv(TABLES / 'model_metrics_natural.csv')
common = pd.read_csv(TABLES / 'model_metrics_common.csv')
common[['split', 'model', 'observation_count', 'mae_mwh', 'rmse_mwh', 'mape_pct', 'r_squared']]


## Why beating persistence matters

One-hour demand is strongly related to the previous hour, so persistence is a difficult and meaningful benchmark. A model with high R² is not automatically useful: it must reduce error relative to persistence on identical timestamps. Positive percentages below mean lower MAE than persistence.

In [ ]:
comparison = common.copy()
persistence_mae = comparison.query("model == 'persistence_1h'").set_index('split')['mae_mwh']
comparison['mae_improvement_vs_persistence_pct'] = comparison.apply(
    lambda row: 100 * (persistence_mae[row['split']] - row['mae_mwh']) / persistence_mae[row['split']],
    axis=1,
)
comparison[['split', 'model', 'mae_mwh', 'mae_improvement_vs_persistence_pct']]

## Peak demand and validation-to-test change

Peak and low-demand thresholds are calculated independently inside each evaluation split. The tables below use the common timestamps for persistence and the selected Linear Regression and XGBoost variants.

In [ ]:
peak = pd.read_csv(TABLES / 'demand_percentile_performance.csv')
degradation = pd.read_csv(TABLES / 'validation_to_test_degradation.csv')
display(peak.query("split == 'test' and demand_segment == 'top_10_percent'")[[
    'model', 'observation_count', 'threshold_mwh', 'mae_mwh', 'rmse_mwh', 'mean_error_bias_mwh'
]])
degradation[['model', 'validation_mae_mwh', 'test_mae_mwh', 'mae_change_pct']]

## One step is not 24 recursive steps

This experiment receives newly observed demand history before every one-hour-ahead prediction. A recursive 24-hour forecast cannot observe future lagged demand within its horizon; it must feed earlier predictions back as inputs. Those errors can accumulate. Strong one-step results justify a separate recursive experiment, but they do not guarantee that the recursive forecast will be strong.

In [ ]:
validation = pd.read_csv(TABLES / 'one_step_validation_results.csv')
validation[['check', 'passed']]